In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import re
pd.set_option('display.max_columns', 500)

In [ ]:
df_orig = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data/Final_data.csv")
df_orig = df_orig.iloc[:,2:]

In [ ]:
#aseg code to extract the columns of dataframe [already used and extracted]
root_dir = "/project_cephfs/3022017.06/UKB/freesurfer/"
all_subject_dirs = [
    d for d in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, d)) and re.match(r"^[A-Za-z0-9_\-]+$", d)
]

all_data = []

for subj in all_subject_dirs:
    subj_dir = os.path.join(root_dir, subj)
    aseg_path = os.path.join(subj_dir, "stats", "aseg.stats")

    if os.path.exists(aseg_path):
        aseg_data = pd.read_csv(aseg_path, sep=r"\s+", comment="#", header=None)
        aseg_data = aseg_data.drop(aseg_data.columns[:2], axis=1)

        aseg_data.columns = [
            "NVoxels", "Volume_mm3", "StructName",
            "normMean", "normStdDev", "normMin", "normMax", "normRange"
        ]

        pivoted_data = aseg_data.pivot_table(
            index=None,
            columns="StructName",
            values= ["Volume_mm3"]
        )

        pivoted_data["SubjectID"] = subj
        pivoted_data = pivoted_data.set_index("SubjectID")

        all_data.append(pivoted_data)

final_df = pd.concat(all_data, axis=0)

print("\nFinal combined DataFrame:")
print(final_df.head())

#final_df.to_csv("correct_aseg_stats.csv")


In [ ]:
df_orig = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data/Final_data.csv")
df_new = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/data_prep/volume_mm3_aseg_stats.csv")
df_orig = df_orig.iloc[:,2:]
df_orig["SubjectID_new"] = df_orig["SubjectID"]
df_orig = df_orig.iloc[:,45:]
df_orig = df_orig.rename(columns={"SubjectID_new":"SubjectID"})

df_orig["SubjectID"] = df_orig["SubjectID"].astype(str)
df_orig = df_orig[df_orig["SubjectID"].str.len() <= 8]

df_orig

In [ ]:
df_new["SubjectID"] = df_new["SubjectID"].astype(str)
df_new = df_new[df_new["SubjectID"].str.len() <= 8]
df_new

In [ ]:
df_orig["SubjectID"] = df_orig["SubjectID"].astype(int)
df_new["SubjectID"] = df_new["SubjectID"].astype(int)

df_orig = df_orig.merge(df_new, on="SubjectID", how="left", suffixes=('', ''))


In [ ]:
df_orig.to_csv("Final_data_correct.csv")

In [ ]:
df_oasis = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/data_prep/Oasis3_data.csv")
df_oasis